**Load the cleaned dataset and inspect what RDKit can compute out of the box**

In [1]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors

df = pd.read_csv("beard_uvvis_cleaned.csv")
print(df.shape)
print(df.head())

# Just check: how many named descriptors does RDKit's Descriptors module expose?
all_descriptors = [name for name, func in Descriptors._descList]
print(f"Total available RDKit descriptors: {len(all_descriptors)}")
print(all_descriptors[:20])

(7167, 5)
                                                 SMI  \
0                    OC[C@@H]([C@H]1OC(=O)C(=C1O)O)O   
1  Oc1cc(O)c2c(c1)O[C@@H]([C@@H](C2)OC(=O)c1cc(O)...   
2                      COC(=O)Cc1ccc(cc1)OC[C@H]1OC1   
3     CN1CCN(CC1)c1c(F)cc2c3c1OCC(n3cc(c2=O)C(=O)O)C   
4                      OC(=O)C(Cn1cnc2c1ncnc2N(C)C)N   

                                       canonical_smi  lambda_max_exp_nm  \
0                    O=C1O[C@H]([C@@H](O)CO)C(O)=C1O              280.0   
1  O=C(O[C@@H]1Cc2c(O)cc(O)cc2O[C@@H]1c1cc(O)c(O)...              280.0   
2                     COC(=O)Cc1ccc(OC[C@@H]2CO2)cc1              225.0   
3   CC1COc2c(N3CCN(C)CC3)c(F)cc3c(=O)c(C(=O)O)cn1c23              820.0   
4                      CN(C)c1ncnc2c1ncn2CC(N)C(=O)O              267.6   

   extinction solvent  
0         NaN     NaN  
1         NaN     NaN  
2         NaN     NaN  
3         NaN     NaN  
4         NaN     NaN  
Total available RDKit descriptors: 217
['MaxAbsEStateIndex

**Molecular Descriptors**

In [2]:
from rdkit.Chem import Descriptors, Lipinski
import numpy as np

baseline_descriptor_names = [
    "MolWt", "MolLogP", "TPSA", "NumHDonors", "NumHAcceptors",
    "NumRotatableBonds", "RingCount", "NumAromaticRings",
    "FractionCSP3", "HeavyAtomCount", "NumValenceElectrons"
]

def compute_baseline_descriptors(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return [np.nan] * len(baseline_descriptor_names)
    values = []
    for name in baseline_descriptor_names:
        func = dict(Descriptors._descList)[name]
        try:
            values.append(func(mol))
        except Exception:
            values.append(np.nan)
    return values

desc_matrix = df["canonical_smi"].apply(compute_baseline_descriptors)
desc_df = pd.DataFrame(desc_matrix.tolist(), columns=baseline_descriptor_names)

df_features = pd.concat([df.reset_index(drop=True), desc_df], axis=1)

print(df_features.shape)
print(df_features[baseline_descriptor_names].isna().sum())
print(df_features[baseline_descriptor_names].describe())

[06:20:16] WARNING: not removing hydrogen atom without neighbors
[06:20:16] WARNING: not removing hydrogen atom without neighbors
[06:20:16] WARNING: not removing hydrogen atom without neighbors
[06:20:16] WARNING: not removing hydrogen atom without neighbors
[06:20:16] WARNING: not removing hydrogen atom without neighbors
[06:20:16] WARNING: not removing hydrogen atom without neighbors
[06:20:16] WARNING: not removing hydrogen atom without neighbors
[06:20:16] WARNING: not removing hydrogen atom without neighbors
[06:20:16] WARNING: not removing hydrogen atom without neighbors
[06:20:16] WARNING: not removing hydrogen atom without neighbors
[06:20:16] WARNING: not removing hydrogen atom without neighbors
[06:20:19] WARNING: not removing hydrogen atom without neighbors
[06:20:19] WARNING: not removing hydrogen atom without neighbors
[06:20:19] WARNING: not removing hydrogen atom without neighbors
[06:20:19] WARNING: not removing hydrogen atom without neighbors
[06:20:19] WARNING: not r

(7167, 16)
MolWt                  0
MolLogP                0
TPSA                   0
NumHDonors             0
NumHAcceptors          0
NumRotatableBonds      0
RingCount              0
NumAromaticRings       0
FractionCSP3           0
HeavyAtomCount         0
NumValenceElectrons    0
dtype: int64
             MolWt      MolLogP         TPSA   NumHDonors  NumHAcceptors  \
count  7167.000000  7167.000000  7167.000000  7167.000000    7167.000000   
mean    409.240532     4.981474    62.331172     0.929120       4.014511   
std     250.748427     4.782585    67.367229     2.256939       3.372793   
min       1.008000   -58.613590     0.000000     0.000000       0.000000   
25%     258.559000     2.169500    26.300000     0.000000       2.000000   
50%     373.302000     4.348100    52.990000     0.000000       4.000000   
75%     500.529000     6.912450    82.260000     1.000000       5.000000   
max    6203.573000    77.424000  2936.620000   109.000000     106.000000   

       NumRotata

[06:20:19] WARNING: not removing hydrogen atom without neighbors
[06:20:19] WARNING: not removing hydrogen atom without neighbors
[06:20:19] WARNING: not removing hydrogen atom without neighbors
[06:20:19] WARNING: not removing hydrogen atom without neighbors
[06:20:19] WARNING: not removing hydrogen atom without neighbors
[06:20:19] WARNING: not removing hydrogen atom without neighbors


**Inspect the extremes directly before deciding a cutoff**

In [3]:
# Look at the largest molecules by weight
print(df_features.nlargest(10, "MolWt")[["canonical_smi", "MolWt", "HeavyAtomCount", "lambda_max_exp_nm"]])

# Look at the smallest / degenerate molecules
print(df_features.nsmallest(10, "MolWt")[["canonical_smi", "MolWt", "HeavyAtomCount", "lambda_max_exp_nm"]])

                                          canonical_smi     MolWt  \
6742  CC[C@H](C)[C@H](NC(=O)CNC(=O)[C@@H]1CN1C(=O)[C...  6203.573   
5726  COc1cc(C(OCc2cc(COC(c3cc(OC)c(OCc4cc(OCc5cc(OC...  5574.207   
6630  CC[C@H](C)[C@H](NC(=O)[C@H](CCCNC(=N)N)NC(=O)[...  3897.856   
6631  CC[C@H](C)[C@H](NC(=O)[C@H](CCCNC(=N)N)NC(=O)[...  3854.827   
5438  CC1c2cc(-c3nc4cc(C=Cc5ccc(N(c6ccc(C=Cc7ccc8nc(...  2850.646   
5143  OC[C@H]1O[C@@H](Oc2ccc(-c3c4nc(c(-c5ccc(O[C@@H...  2370.407   
5142  OC[C@H]1O[C@@H](Oc2ccc(-c3c4nc(c(-c5ccc(O[C@@H...  2370.407   
5145  OC[C@H]1O[C@@H](Oc2ccc(-c3c4nc(c(-c5ccc(O[C@@H...  2356.404   
5374  CC1c2cc(C#Cc3ccc4c(c3)C(C)(C)c3ccccc3-4)ccc2-c...  2175.829   
5144  Cc1ccccc1-c1c2nc(c(-c3ccccc3C)c3ccc([nH]3)c(-c...  1878.068   

      HeavyAtomCount  lambda_max_exp_nm  
6742             427              201.0  
5726             417              339.0  
6630             268              502.0  
6631             265              498.0  
5438             222          

**defensible on structural-domain grounds, not arbitrary**

In [4]:
# Lower bound: exclude fragments/isolated atoms/ions (no meaningful conjugated system possible)
# Upper bound: MW 1000 Da is a conventional, citable boundary for "small molecule" chemical space
# (also standard in drug-discovery/cheminformatics as the small-molecule regime)

before = df_features.shape[0]
df_scoped = df_features[
    (df_features["HeavyAtomCount"] >= 3) &
    (df_features["MolWt"] <= 1000)
].copy()
after = df_scoped.shape[0]

print(f"Rows before scoping: {before}")
print(f"Rows after scoping: {after}")
print(f"Rows removed: {before - after}")

print(df_scoped[["MolWt", "HeavyAtomCount"]].describe())

Rows before scoping: 7167
Rows after scoping: 6878
Rows removed: 289
             MolWt  HeavyAtomCount
count  6878.000000     6878.000000
mean    398.236935       28.231026
std     188.686203       13.996073
min      39.683000        3.000000
25%     265.355500       18.000000
50%     373.918000       26.000000
75%     492.568000       35.000000
max     999.186000       78.000000


**check the remaining baseline descriptors for any lingering artifacts before we finalize**

In [5]:
print(df_scoped[["MolLogP", "TPSA", "NumHDonors", "NumHAcceptors",
                  "NumRotatableBonds", "RingCount", "NumAromaticRings",
                  "FractionCSP3", "NumValenceElectrons"]].describe())

           MolLogP         TPSA   NumHDonors  NumHAcceptors  \
count  6878.000000  6878.000000  6878.000000    6878.000000   
mean      4.859600    61.823239     0.899099       3.990259   
std       4.098612    47.596988     1.351649       2.704117   
min     -21.066000     0.000000     0.000000       0.000000   
25%       2.312950    29.155000     0.000000       2.000000   
50%       4.369940    54.180000     0.000000       4.000000   
75%       6.803400    82.460000     1.000000       5.000000   
max      21.744640   407.740000    13.000000      24.000000   

       NumRotatableBonds    RingCount  NumAromaticRings  FractionCSP3  \
count        6878.000000  6878.000000       6878.000000   6878.000000   
mean            4.246438     4.019482          3.443588      0.183632   
std             3.818846     2.613969          2.476518      0.219312   
min             0.000000     0.000000          0.000000      0.000000   
25%             2.000000     2.000000          2.000000      0.0312

**Check the LogP extremes directly, same pattern as before**

In [6]:
print(df_scoped.nlargest(5, "MolLogP")[["canonical_smi", "MolLogP", "MolWt", "lambda_max_exp_nm"]])
print(df_scoped.nsmallest(5, "MolLogP")[["canonical_smi", "MolLogP", "MolWt", "lambda_max_exp_nm"]])

                                          canonical_smi   MolLogP    MolWt  \
5874  Cc1c2oc(-c3ccc(-c4ccc5ccccc5c4)cc3)c(-c3ccc(-c...  21.74464  995.234   
5875  Cc1c2oc(-c3ccc(-c4cccc5ccccc45)cc3)c(-c3ccc(-c...  21.74464  995.234   
5495  CC1c2cc(-c3ccc(-c4ccc(-c5ccc(-c6ccc(-c7ccc8ccc...  20.98930  986.409   
6833  Cc1ccc(N(c2ccc(C)cc2)c2cc(N(c3ccc(C)cc3)c3ccc(...  20.93056  983.316   
5364  CCn1c2ccc(C=Cc3ccc4c(c3)c3cc(N(c5ccccc5)c5ccc6...  20.84700  997.258   

      lambda_max_exp_nm  
5874              259.0  
5875              292.0  
5495              440.0  
6833              476.0  
5364              356.0  
                                          canonical_smi  MolLogP    MolWt  \
5303  O=P1([O-])OP(=O)([O-])OP(=O)([O-])OP(=O)([O-])... -21.0660  611.766   
6451  O=P([O-])([O-])OP(=O)([O-])[O-].[Na+].[Na+].[N... -15.3236  265.901   
5968  O=C([O-])CC(O)(CC(=O)[O-])C(=O)[O-].[Na+].[Na+... -14.2406  258.069   
5902  O=C([O-])CCN(CCC(=O)[O-])c1ccc(C=C2CC(=Cc3ccc(... -13.6848  6

In [7]:
df_scoped["MolLogP_clipped"] = df_scoped["MolLogP"].clip(lower=-5, upper=15)

print(df_scoped[["MolLogP", "MolLogP_clipped"]].describe())

           MolLogP  MolLogP_clipped
count  6878.000000      6878.000000
mean      4.859600         4.830044
std       4.098612         3.889492
min     -21.066000        -5.000000
25%       2.312950         2.312950
50%       4.369940         4.369940
75%       6.803400         6.803400
max      21.744640        15.000000


**Save the Phase 2 baseline-track output**

In [8]:
# Keep original MolLogP for reference, but MolLogP_clipped is what we'll actually feed the baseline model
baseline_cols = ["SMI", "canonical_smi", "lambda_max_exp_nm"] + baseline_descriptor_names + ["MolLogP_clipped"]

df_baseline_export = df_scoped[baseline_cols].copy()
df_baseline_export.to_csv("beard_baseline_features.csv", index=False)
print(f"Saved {df_baseline_export.shape[0]} rows, {df_baseline_export.shape[1]} columns to beard_baseline_features.csv")

Saved 6878 rows, 15 columns to beard_baseline_features.csv


**Checking Conjugation and Push Pull**

In [9]:
from collections import defaultdict, deque

donor_smarts = {
    "OH": "[OX2H]",
    "OR": "[OX2H0;!$(O=*)]",
    "Amine": "[NX3;!$(N=O);!$(N-C=O)]",
    "SR": "[#16X2H0]",
}

acceptor_smarts = {
    "NO2": "[NX3](=O)=O",
    "CN": "[CX2]#[NX1]",
    "C=O": "[CX3]=[OX1]",
    "SO2": "[SX4](=O)(=O)",
    "CF3": "[CX4](F)(F)F",
    "AromHalogen": "[c][F,Cl,Br,I]",
}

donor_patterns = {k: Chem.MolFromSmarts(v) for k, v in donor_smarts.items()}
acceptor_patterns = {k: Chem.MolFromSmarts(v) for k, v in acceptor_smarts.items()}

def get_conjugation_features(mol):
    conj_bonds = [b for b in mol.GetBonds() if b.GetIsConjugated()]
    num_conj_bonds = len(conj_bonds)

    adj = defaultdict(set)
    for b in conj_bonds:
        a1, a2 = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        adj[a1].add(a2)
        adj[a2].add(a1)

    visited = set()
    largest = 0
    for start in adj:
        if start not in visited:
            queue = deque([start])
            visited.add(start)
            size = 0
            while queue:
                cur = queue.popleft()
                size += 1
                for nb in adj[cur]:
                    if nb not in visited:
                        visited.add(nb)
                        queue.append(nb)
            largest = max(largest, size)

    return num_conj_bonds, largest

def get_donor_acceptor_counts(mol):
    n_donor = sum(len(mol.GetSubstructMatches(p)) for p in donor_patterns.values())
    n_acceptor = sum(len(mol.GetSubstructMatches(p)) for p in acceptor_patterns.values())
    return n_donor, n_acceptor

def compute_mechanism_descriptors(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return [np.nan] * 5
    num_conj_bonds, largest_conj = get_conjugation_features(mol)
    n_donor, n_acceptor = get_donor_acceptor_counts(mol)
    has_push_pull = int(n_donor > 0 and n_acceptor > 0)
    return [num_conj_bonds, largest_conj, n_donor, n_acceptor, has_push_pull]

mech_cols = ["NumConjugatedBonds", "LargestConjugatedSystemSize", "NumDonorGroups", "NumAcceptorGroups", "HasPushPull"]

mech_matrix = df_scoped["canonical_smi"].apply(compute_mechanism_descriptors)
mech_df = pd.DataFrame(mech_matrix.tolist(), columns=mech_cols)

df_mech = pd.concat([df_scoped.reset_index(drop=True), mech_df], axis=1)

print(df_mech.shape)
print(df_mech[mech_cols].describe())
print(df_mech[mech_cols].isna().sum())

[11:10:54] WARNING: not removing hydrogen atom without neighbors
[11:10:54] WARNING: not removing hydrogen atom without neighbors
[11:10:54] WARNING: not removing hydrogen atom without neighbors
[11:10:54] WARNING: not removing hydrogen atom without neighbors
[11:10:54] WARNING: not removing hydrogen atom without neighbors
[11:10:54] WARNING: not removing hydrogen atom without neighbors
[11:10:54] WARNING: not removing hydrogen atom without neighbors
[11:10:54] WARNING: not removing hydrogen atom without neighbors
[11:10:54] WARNING: not removing hydrogen atom without neighbors
[11:10:54] WARNING: not removing hydrogen atom without neighbors


(6878, 22)
       NumConjugatedBonds  LargestConjugatedSystemSize  NumDonorGroups  \
count         6878.000000                  6878.000000     6878.000000   
mean            25.478046                    20.652224        1.849520   
std             15.578777                    13.006973        1.889689   
min              0.000000                     0.000000        0.000000   
25%             15.000000                    12.000000        0.000000   
50%             24.000000                    19.000000        2.000000   
75%             33.000000                    27.000000        3.000000   
max             94.000000                    78.000000       18.000000   

       NumAcceptorGroups  HasPushPull  
count        6878.000000  6878.000000  
mean            1.088398     0.457837  
std             1.289675     0.498255  
min             0.000000     0.000000  
25%             0.000000     0.000000  
50%             1.000000     0.000000  
75%             2.000000     1.000000  
ma

[11:10:55] WARNING: not removing hydrogen atom without neighbors
[11:10:55] WARNING: not removing hydrogen atom without neighbors
[11:10:55] WARNING: not removing hydrogen atom without neighbors
[11:10:55] WARNING: not removing hydrogen atom without neighbors
[11:10:55] WARNING: not removing hydrogen atom without neighbors
[11:10:55] WARNING: not removing hydrogen atom without neighbors
[11:10:55] WARNING: not removing hydrogen atom without neighbors
[11:10:55] WARNING: not removing hydrogen atom without neighbors
[11:10:55] WARNING: not removing hydrogen atom without neighbors
[11:10:55] WARNING: not removing hydrogen atom without neighbors
[11:10:55] WARNING: not removing hydrogen atom without neighbors
[11:10:55] WARNING: not removing hydrogen atom without neighbors
[11:10:55] WARNING: not removing hydrogen atom without neighbors
[11:10:55] WARNING: not removing hydrogen atom without neighbors
[11:10:55] WARNING: not removing hydrogen atom without neighbors
[11:10:55] WARNING: not r

In [10]:
inconsistent = df_mech[df_mech["LargestConjugatedSystemSize"] > df_mech["NumConjugatedBonds"] + 1]
print(f"Rows where largest conjugated system exceeds total conjugated bonds: {inconsistent.shape[0]}")

Rows where largest conjugated system exceeds total conjugated bonds: 0


In [11]:
print(df_mech.groupby("HasPushPull")["lambda_max_exp_nm"].agg(["mean", "median", "count"]))

                   mean  median  count
HasPushPull                           
0            400.882292   377.0   3729
1            419.268694   412.0   3149


**Checking the effect of conjugation**

In [12]:
corr_conj = df_mech["LargestConjugatedSystemSize"].corr(df_mech["lambda_max_exp_nm"])
print(f"Correlation (LargestConjugatedSystemSize vs lambda_max): {corr_conj:.3f}")

# Also bin it into ranges to see the trend more intuitively than a raw correlation number
df_mech["conj_size_bin"] = pd.cut(df_mech["LargestConjugatedSystemSize"], bins=[0, 10, 20, 30, 40, 100])
print(df_mech.groupby("conj_size_bin")["lambda_max_exp_nm"].agg(["mean", "median", "count"]))

Correlation (LargestConjugatedSystemSize vs lambda_max): 0.162
                     mean  median  count
conj_size_bin                           
(0, 10]        377.964749  331.00   1266
(10, 20]       389.743033  367.00   2327
(20, 30]       426.082218  413.00   1790
(30, 40]       440.816762  418.55    698
(40, 100]      452.856092  438.00    539


/var/folders/4t/nmlmkdbd4xq3p07cyv3s7qt00000gn/T/ipykernel_32463/4145564987.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df_mech.groupby("conj_size_bin")["lambda_max_exp_nm"].agg(["mean", "median", "count"]))


**No. of donor and no. of accptor**

In [14]:
corr_donor = df_mech["NumDonorGroups"].corr(df_mech["lambda_max_exp_nm"])
corr_acceptor = df_mech["NumAcceptorGroups"].corr(df_mech["lambda_max_exp_nm"])
print(f"Correlation (NumDonorGroups vs lambda_max): {corr_donor:.3f}")
print(f"Correlation (NumAcceptorGroups vs lambda_max): {corr_acceptor:.3f}")

print(df_mech.groupby("NumDonorGroups")["lambda_max_exp_nm"].agg(["mean", "count"]))
print(df_mech.groupby("NumAcceptorGroups")["lambda_max_exp_nm"].agg(["mean", "count"]))

Correlation (NumDonorGroups vs lambda_max): 0.128
Correlation (NumAcceptorGroups vs lambda_max): 0.098
                      mean  count
NumDonorGroups                   
0               386.843588   1770
1               393.013831   1662
2               422.965923   1634
3               438.690424    771
4               422.286997    545
5               443.357219    187
6               439.139552    134
7               462.863462     52
8               410.311765     51
9               402.104762     21
10              460.916667     18
11              413.000000     10
12              457.272727     11
13              433.000000      5
14              376.500000      2
15              397.000000      1
16              534.500000      2
18              365.000000      2
                         mean  count
NumAcceptorGroups                   
0                  405.786942   2954
1                  391.136567   1720
2                  419.008449   1459
3                  451.513934   

**Correlation matrix among all mechanism (and baseline) features**

In [15]:
feature_cols_to_check = [
    "MolWt", "MolLogP_clipped", "TPSA", "NumHDonors", "NumHAcceptors",
    "NumRotatableBonds", "RingCount", "NumAromaticRings", "FractionCSP3",
    "HeavyAtomCount", "NumValenceElectrons",
    "NumConjugatedBonds", "LargestConjugatedSystemSize",
    "NumDonorGroups", "NumAcceptorGroups", "HasPushPull"
]

corr_matrix = df_mech[feature_cols_to_check].corr()
print(corr_matrix.round(2))

                             MolWt  MolLogP_clipped  TPSA  NumHDonors  \
MolWt                         1.00             0.73  0.22        0.01   
MolLogP_clipped               0.73             1.00 -0.29       -0.35   
TPSA                          0.22            -0.29  1.00        0.67   
NumHDonors                    0.01            -0.35  0.67        1.00   
NumHAcceptors                 0.36            -0.09  0.86        0.44   
NumRotatableBonds             0.63             0.37  0.36        0.16   
RingCount                     0.81             0.81 -0.04       -0.14   
NumAromaticRings              0.73             0.81 -0.09       -0.20   
FractionCSP3                 -0.11            -0.29  0.11        0.22   
HeavyAtomCount                0.97             0.78  0.21        0.00   
NumValenceElectrons           0.97             0.75  0.25        0.04   
NumConjugatedBonds            0.83             0.82  0.06       -0.14   
LargestConjugatedSystemSize   0.72             0.77

In [16]:
import numpy as np

# Get upper triangle of correlation matrix (avoid duplicate pairs and the diagonal)
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Find any pair above a defensible redundancy threshold
strong_pairs = upper.stack()
strong_pairs = strong_pairs[strong_pairs.abs() > 0.75].sort_values(ascending=False)
print(strong_pairs)

HeavyAtomCount       NumValenceElectrons            0.993685
MolWt                NumValenceElectrons            0.974579
                     HeavyAtomCount                 0.969711
RingCount            NumAromaticRings               0.938094
NumAromaticRings     NumConjugatedBonds             0.936342
RingCount            NumConjugatedBonds             0.930236
NumConjugatedBonds   LargestConjugatedSystemSize    0.924584
HeavyAtomCount       NumConjugatedBonds             0.898043
RingCount            HeavyAtomCount                 0.865201
TPSA                 NumHAcceptors                  0.859527
NumAromaticRings     LargestConjugatedSystemSize    0.851153
NumValenceElectrons  NumConjugatedBonds             0.847106
MolWt                NumConjugatedBonds             0.829985
RingCount            LargestConjugatedSystemSize    0.827817
MolLogP_clipped      NumConjugatedBonds             0.821771
RingCount            NumValenceElectrons            0.817003
MolLogP_clipped      Num

**one final, non-redundant feature set by picking one representative per cluster**

In [17]:
final_feature_cols = [
    "HeavyAtomCount", "LargestConjugatedSystemSize", "TPSA",
    "NumHDonors", "NumRotatableBonds", "FractionCSP3", "MolLogP_clipped",
    "NumDonorGroups", "NumAcceptorGroups", "HasPushPull"
]

df_model_ready = df_mech[["SMI", "canonical_smi", "lambda_max_exp_nm"] + final_feature_cols].copy()
df_model_ready.to_csv("beard_model_ready_features.csv", index=False)
print(df_model_ready.shape)
print(df_model_ready[final_feature_cols].corr().round(2))

(6878, 13)
                             HeavyAtomCount  LargestConjugatedSystemSize  \
HeavyAtomCount                         1.00                         0.79   
LargestConjugatedSystemSize            0.79                         1.00   
TPSA                                   0.21                         0.03   
NumHDonors                             0.00                        -0.16   
NumRotatableBonds                      0.61                         0.26   
FractionCSP3                          -0.16                        -0.46   
MolLogP_clipped                        0.78                         0.77   
NumDonorGroups                         0.28                         0.05   
NumAcceptorGroups                      0.16                         0.02   
HasPushPull                            0.08                        -0.02   

                             TPSA  NumHDonors  NumRotatableBonds  \
HeavyAtomCount               0.21        0.00               0.61   
LargestConjugate